In [20]:
import os
import subprocess
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================
# INPUT AND OUTPUT FILES
# ============================================================
precip_file = "monthly_MSWEP_1980_2022_025.nc"
mask_file = "cluster_1.nc"

spi_output = "cluster_1_spi_MSWEP_Africa_1980_2022.csv"
precip_output = "cluster_1_pre_MSWEP_Africa_1980_2022.csv"

# Folder where the R script will be saved
r_script_directory = r"C:/Users/Milic/Desktop/Codigos/data/SPI_data"
r_script_name = "calculate_spi_temp.R"
r_script_path = os.path.join(r_script_directory, r_script_name)

# Rscript executable
rscript_exe = "C:/Program Files/R/R-4.4.1/bin/x64/Rscript.exe"

# ============================================================
# CREATE DIRECTORY IF NEEDED
# ============================================================
os.makedirs(r_script_directory, exist_ok=True)

# ============================================================
# READ DATA
# ============================================================
ds_precip = xr.open_dataset(precip_file)
ds_mask = xr.open_dataset(mask_file)

precip = ds_precip["precipitation"]
mask = ds_mask["mask"]

print("Precipitation dimensions:", precip.dims, precip.shape)
print("Mask dimensions:", mask.dims, mask.shape)

# ============================================================
# IDENTIFY DIMENSIONS
# ============================================================
time_dim = [d for d in precip.dims if d.lower() == "time"][0]
lon_dim = [d for d in precip.dims if d.lower() in ["lon", "longitude"]][0]
lat_dim = [d for d in precip.dims if d.lower() in ["lat", "latitude"]][0]

# Reorder dimensions if needed
precip = precip.transpose(lon_dim, lat_dim, time_dim)
mask = mask.transpose(lon_dim, lat_dim)

# ============================================================
# COMPUTE REGIONAL MEAN WITHOUT BUILDING A 3D MASKED ARRAY
# ============================================================
mask_np = mask.values
mask_valid = mask_np > 0

n_time = precip.sizes[time_dim]
mean_precip = np.full(n_time, np.nan, dtype=float)

for t in range(n_time):
    print(f"Processing month {t+1}/{n_time}", end="\r")
    precip_t = precip.isel({time_dim: t}).values
    mean_precip[t] = np.nanmean(precip_t[mask_valid])

print("\nRegional mean precipitation computed.")

# ============================================================
# SAVE REGIONAL PRECIPITATION SERIES
# ============================================================
pd.DataFrame({"precipitation": mean_precip}).to_csv(
    precip_output,
    sep=";",
    index=False,
    float_format="%.8f"
)

print(f"Regional precipitation series saved to: {precip_output}")

# ============================================================
# CREATE R SCRIPT TO CALCULATE SPI
# ============================================================
precip_path_r = os.path.abspath(precip_output).replace("\\", "/")
spi_path_r = os.path.abspath(spi_output).replace("\\", "/")

r_code = f'''
library(SPEI)

input_file  <- "{precip_path_r}"
output_file <- "{spi_path_r}"

data <- read.csv(input_file, sep=";")
precip_series <- data$precipitation

spi_matrix <- matrix(NA, nrow=length(precip_series), ncol=48)

for (i in 1:48) {{
  spi_matrix[, i] <- spi(
    data = ts(precip_series, freq=12, start=c(1980,1)),
    scale = i,
    ref.start = c(1980,1),
    ref.end   = c(2022,12),
    distribution = "Gamma"
  )$fitted
}}

write.table(spi_matrix, output_file, sep=";", row.names=FALSE, col.names=FALSE)
cat("SPI saved to:", output_file, "\\n")
'''

# ============================================================
# SAVE R SCRIPT
# ============================================================
with open(r_script_path, "w", encoding="utf-8") as f:
    f.write(r_code)

print(f"R script saved at: {r_script_path}")

# ============================================================
# RUN RSCRIPT
# ============================================================
result = subprocess.run(
    [rscript_exe, r_script_path],
    capture_output=True,
    text=True
)

print("\n========== R STDOUT ==========")
print(result.stdout)

print("\n========== R STDERR ==========")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Rscript execution failed.\n"
        "Check that R is installed, Rscript is in the PATH, and the SPEI package is installed."
    )

print(f"\nSPI results saved to: {spi_output}")

Precipitation dimensions: ('time', 'lat', 'lon') (516, 720, 1440)
Mask dimensions: ('lat', 'lon') (720, 1440)
Processing month 516/516
Regional mean precipitation computed.
Regional precipitation series saved to: cluster_1_pre_MSWEP_Africa_1980_2022.csv
R script saved at: C:/Users/Milic/Desktop/Codigos/data/SPI_data\calculate_spi_temp.R

========== R STDOUT ==========
[1] "Calculating the Standardized Precipitation Evapotranspiration Index (SPEI) at a time scale of 1. Using kernel type 'rectangular', with 0 shift. Fitting the data to a Gamma distribution. Using the ub-pwm parameter fitting method. Checking for missing values (`NA`): all the data must be complete. Using a user-specified reference period. Input type is tsvector. Time series spanning Jan 1980 to Dec 2022, with frequency = 12."
[1] "Calculating the Standardized Precipitation Evapotranspiration Index (SPEI) at a time scale of 2. Using kernel type 'rectangular', with 0 shift. Fitting the data to a Gamma distribution. Using t